# Data Wrangling
Note: Some data wrangling processes such as merging datasets and engineering new features such as total_respiratory_admissions were conducted during milestone 1's data acquisition step. The code in the section focuses on data validation through handling duplicates, missing values, and producing the final analysis-ready dataset.

### Imports and loading the data

In [17]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

db_path = PROCESSED_DIR / "milestone1.db"
conn = sqlite3.connect(db_path)

df = pd.read_sql_query("SELECT * FROM merged_m1", conn)
conn.close()

df["week"] = pd.to_datetime(df["week"])
df.head()

,state,week,covid_admissions,influenza_admissions,rsv_admissions,total_respiratory_admissions,aqi_mean,aqi_p90,aqi_max,days_reported,category,defining_parameter
0,AL,2021-10-09,1230.00,8.50,NaN,1238.50,34.580645,52.6,80,93,Good,Ozone
1,AL,2021-10-16,962.57,4.00,NaN,966.57,40.752577,57.0,67,97,Good,PM2.5
2,AL,2021-10-23,707.91,8.00,NaN,715.91,40.423913,55.0,65,92,Good,Ozone
3,AL,2021-10-30,554.86,8.86,NaN,563.72,30.712766,44.4,70,94,Good,Ozone
4,AL,2021-11-06,451.71,10.14,NaN,461.85,38.166667,52.0,57,66,Good,PM2.5


### Data Validation: Duplicates

In [18]:
# Check for duplicates
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate state-week keys:", df.duplicated(subset=["state", "week"]).sum())

Full-row duplicates: 0
Duplicate state-week keys: 0


### Missing Values

In [19]:
# Calculation of value missing values and percentage rates(null values) for applicable variables
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": df.isnull().mean() * 100
}).sort_values("missing_pct", ascending=False)

missing_summary

,missing_count,missing_pct
rsv_admissions,8706,66.559633
influenza_admissions,55,0.420489
covid_admissions,55,0.420489
week,0,0.000000
state,0,0.000000
total_respiratory_admissions,0,0.000000
aqi_mean,0,0.000000
aqi_p90,0,0.000000
aqi_max,0,0.000000
days_reported,0,0.000000


### Cleaning Decisions

In [20]:
# Drop rows missing the modeling target
df = df.dropna(subset=["total_respiratory_admissions"]).copy()

# Drop rows where no AQI values were reported that week
df = df[df["days_reported"] > 0].copy()

# fill disease-specific admissions with 0 to keep them
for col in ["covid_admissions", "influenza_admissions", "rsv_admissions"]:
    df[col] = df[col].fillna(0)

### Sort the data

In [21]:
df = df.sort_values(["state", "week"]).reset_index(drop=True)
df.head()

,state,week,covid_admissions,influenza_admissions,rsv_admissions,total_respiratory_admissions,aqi_mean,aqi_p90,aqi_max,days_reported,category,defining_parameter
0,AK,2021-01-02,68.57,0.0,0.0,68.57,74.692308,156.8,189,13,Good,PM2.5
1,AK,2021-01-09,77.07,0.0,0.0,77.07,44.414634,116.0,174,41,Good,PM2.5
2,AK,2021-01-16,59.71,0.0,0.0,59.71,36.651163,75.8,163,43,Good,PM2.5
3,AK,2021-01-23,46.43,0.0,0.0,46.43,38.000000,87.0,116,41,Good,PM2.5
4,AK,2021-01-30,41.57,0.0,0.0,41.57,45.975610,86.0,163,41,Good,PM2.5


### Feature engineering: time features

In [22]:
df["year"] = df["week"].dt.year
df["month"] = df["week"].dt.month
df["week_of_year"] = df["week"].dt.isocalendar().week.astype(int)
df["quarter"] = df["week"].dt.quarter

### Feature engineering: lag features

In [23]:
df["aqi_mean_lag1"] = df.groupby("state")["aqi_mean"].shift(1)
df["aqi_mean_lag2"] = df.groupby("state")["aqi_mean"].shift(2)

df["aqi_max_lag1"] = df.groupby("state")["aqi_max"].shift(1)
df["aqi_max_lag2"] = df.groupby("state")["aqi_max"].shift(2)

df["aqi_p90_lag1"] = df.groupby("state")["aqi_p90"].shift(1)
df["aqi_p90_lag2"] = df.groupby("state")["aqi_p90"].shift(2)

In [24]:
df["aqi_mean_rolling_3"] = (
    df.groupby("state")["aqi_mean"]
      .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

### Handle lag-induced missingness

In [25]:
model_df = df.dropna(subset=[
    "aqi_mean_lag1", "aqi_mean_lag2",
    "aqi_max_lag1", "aqi_max_lag2",
    "aqi_p90_lag1", "aqi_p90_lag2"
]).copy()

model_df.shape

(12974, 23)

### Final feature selection

In [26]:
model_df = model_df[[
    "state", "week",
    "total_respiratory_admissions",
    "aqi_mean", "aqi_p90", "aqi_max", "days_reported",
    "year", "month", "week_of_year", "quarter",
    "aqi_mean_lag1", "aqi_mean_lag2",
    "aqi_max_lag1", "aqi_max_lag2",
    "aqi_p90_lag1", "aqi_p90_lag2",
    "aqi_mean_rolling_3"
]].copy()

model_df.head()

,state,week,total_respiratory_admissions,aqi_mean,aqi_p90,aqi_max,days_reported,year,month,week_of_year,quarter,aqi_mean_lag1,aqi_mean_lag2,aqi_max_lag1,aqi_max_lag2,aqi_p90_lag1,aqi_p90_lag2,aqi_mean_rolling_3
2,AK,2021-01-16,59.71,36.651163,75.8,163,43,2021,1,2,1,44.414634,74.692308,174.0,189.0,116.0,156.8,51.919368
3,AK,2021-01-23,46.43,38.000000,87.0,116,41,2021,1,3,1,36.651163,44.414634,163.0,174.0,75.8,116.0,39.688599
4,AK,2021-01-30,41.57,45.975610,86.0,163,41,2021,1,4,1,38.000000,36.651163,116.0,163.0,87.0,75.8,40.208924
5,AK,2021-02-06,39.86,38.045455,64.4,93,44,2021,2,5,1,45.975610,38.000000,163.0,116.0,86.0,87.0,40.673688
6,AK,2021-02-13,33.43,46.475000,69.3,131,40,2021,2,6,1,38.045455,45.975610,93.0,163.0,64.4,86.0,43.498688


### Export clean modeling dataset

In [27]:
model_df.to_csv(PROCESSED_DIR / "modeling_dataset.csv", index=False)
print("Saved modeling dataset.")

Saved modeling dataset.


### Save milestone2.db

In [28]:
db2_path = PROCESSED_DIR / "milestone2.db"
conn = sqlite3.connect(db2_path)

model_df.to_sql("modeling_dataset", conn, index=False, if_exists="replace")

conn.close()
print("Saved milestone2.db")

Saved milestone2.db
